# Assistants

[Assistants](https://langchain-ai.github.io/langgraph/concepts/assistants/#resources) 为开发者提供了一种快速简便的方式来修改和版本化代理以进行实验。

## 为图配置参数

我们的 `task_maistro` 图已经配置为使用 assistants！

它定义了一个 `configuration.py` 文件并在图中加载。

我们在图节点内部访问可配置字段（`user_id`、`todo_category`、`task_maistro_role`）。

## 创建 assistants

那么，对于我们一直在构建的 `task_maistro` 应用，assistants 的实际用例是什么？

对我来说，它是为不同类别的任务拥有独立的待办事项列表的能力。

例如，我想要一个用于个人任务的 assistant，另一个用于工作任务的 assistant。

这些可以使用 `todo_category` 和 `task_maistro_role` 可配置字段轻松配置。

![Screenshot 2024-11-18 at 9.35.55 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/673d50597f4e9eae9abf4869_Screenshot%202024-11-19%20at%206.57.01%E2%80%AFPM.png)

In [ ]:
%%capture --no-stderr
%pip install -U langgraph_sdk

这是我们部署图时创建的默认 assistant。

In [ ]:
from langgraph_sdk import get_client
url_for_cli_deployment = "http://localhost:8123"
client = get_client(url=url_for_cli_deployment)

### 个人 assistant

这是我将用来管理个人任务的个人 assistant。

In [ ]:
personal_assistant = await client.assistants.create(
    # "task_maistro" 是我们部署的图的名称
    "task_maistro", 
    config={"configurable": {"todo_category": "personal"}}
)
print(personal_assistant)

让我们更新这个 assistant 以包含我的 `user_id` 方便使用，[创建它的新版本](https://langchain-ai.github.io/langgraph/cloud/how-tos/assistant_versioning/#create-a-new-version-for-your-assistant)。

In [ ]:
task_maistro_role = """你是一个友善且有条理的个人任务助手。你的主要关注点是帮助用户掌握他们的个人任务和承诺。具体来说：

- 帮助跟踪和组织个人任务
- 当提供'todo summary'时：
  1. 按截止日期分组列出所有当前任务（逾期、今天、本周、未来）
  2. 突出显示任何缺少截止日期的任务，并温和地鼓励添加它们
  3. 注意任何看起来重要但缺乏时间估计的任务
- 当新任务没有截止日期时主动询问截止日期
- 保持支持性的语调，同时帮助用户保持责任感
- 根据截止日期和重要性帮助优先排序任务

你的沟通风格应该是鼓励和有帮助的，绝不批判性的。

当任务缺少截止日期时，回应类似"我注意到[任务]还没有截止日期。你想添加一个以帮助我们更好地跟踪吗？"""

configurations = {"todo_category": "personal", 
                  "user_id": "lance",
                  "task_maistro_role": task_maistro_role}

personal_assistant = await client.assistants.update(
    personal_assistant["assistant_id"],
    config={"configurable": configurations}
)
print(personal_assistant)

### 工作 assistant

现在，让我们创建一个工作 assistant。我将用它来处理我的工作任务。

In [ ]:
task_maistro_role = """你是一个专注且高效的工作任务助手。

你的主要关注点是帮助用户用现实的时间框架管理他们的工作承诺。

具体来说：

- 帮助跟踪和组织工作任务
- 当提供'todo summary'时：
  1. 按截止日期分组列出所有当前任务（逾期、今天、本周、未来）
  2. 突出显示任何缺少截止日期的任务，并温和地鼓励添加它们
  3. 注意任何看起来重要但缺乏时间估计的任务
- 当讨论新任务时，建议用户根据任务类型提供现实的时间框架：
  • 开发者关系功能：通常1天
  • 课程课程审查/反馈：通常2天
  • 文档冲刺：通常3天
- 根据截止日期和团队依赖关系帮助优先排序任务
- 保持专业的语调，同时帮助用户保持责任感

你的沟通风格应该是支持性但实用的。

当任务缺少截止日期时，回应类似"我注意到[任务]还没有截止日期。根据类似任务，这可能需要[建议的时间框架]。你想在考虑这个的情况下设置截止日期吗？"""

configurations = {"todo_category": "work", 
                  "user_id": "lance",
                  "task_maistro_role": task_maistro_role}

work_assistant = await client.assistants.create(
    # "task_maistro" 是我们部署的图的名称
    "task_maistro", 
    config={"configurable": configurations}
)
print(work_assistant)

## 使用 assistants

Assistants 将保存到我们部署中的 `Postgres`。

这使我们能够轻松地使用 SDK [搜索](https://langchain-ai.github.io/langgraph/cloud/how-tos/configuration_cloud/) assistants。

In [ ]:
assistants = await client.assistants.search()
for assistant in assistants:
    print({
        'assistant_id': assistant['assistant_id'],
        'version': assistant['version'],
        'config': assistant['config']
    })

我们可以使用 SDK 轻松管理它们。例如，我们可以删除不再使用的 assistants。
> 视频中的语法略有偏差。下面更新的代码创建了一个备用 assistant，然后删除它。

In [ ]:
# 创建一个临时 assistant
temp_assistant = await client.assistants.create(
    "task_maistro", 
    config={"configurable": configurations}
)

assistants = await client.assistants.search()
for assistant in assistants:
    print(f"删除前: {{'assistant_id': {assistant['assistant_id']}}}")
    
# 删除我们的临时 assistant
await client.assistants.delete(assistants[-1]["assistant_id"])
print()

assistants = await client.assistants.search()
for assistant in assistants:
    print(f"删除后: {{'assistant_id': {assistant['assistant_id']} }}")

让我们为我将使用的 `personal` 和 `work` assistants 设置 assistant ID。

In [ ]:
work_assistant_id = assistants[0]['assistant_id']
personal_assistant_id = assistants[1]['assistant_id']

### 工作 assistant

让我们为我的工作 assistant 添加一些待办事项。

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_core.messages import convert_to_messages

user_input = "创建或更新几个待办事项：1) 今天结束前重新拍摄模块6，第5课。2) 下周一前更新audioUX。"
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      work_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

In [ ]:
user_input = "创建另一个待办事项：完成报告生成教程集。"
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      work_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

Assistant 使用它的指令来推进任务创建！

它要求我指定一个截止日期 :)

In [ ]:
user_input = "好的，对于这个任务让我们在下周二完成。"
async for chunk in client.runs.stream(thread["thread_id"], 
                                      work_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

### 个人 assistant

同样，我们可以为我的个人 assistant 添加待办事项。

In [ ]:
user_input = "创建待办事项：1) 这个周末查看婴儿游泳课程。2) 为冬季旅行，检查AmEx积分。"
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      personal_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

In [ ]:
user_input = "给我一个待办事项摘要。"
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      personal_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()